# Security Incident Timeline Tracker

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from matplotlib.patches import Rectangle
import seaborn as sns

class SecurityIncidentTracker:
    def __init__(self):
        self.incidents = []
        self.severity_colors = {
            'Critical': '#FF0000',
            'High': '#FF6600',
            'Medium': '#FFCC00',
            'Low': '#00FF00',
            'Info': '#0066FF'
        }
        self.status_colors = {
            'Open': '#FF4444',
            'In Progress': '#FFAA00',
            'Resolved': '#44FF44',
            'Closed': '#888888'
        }

    def add_incident(self, incident_id, title, severity, category, detected_time,
                    resolved_time=None, status='Open', affected_systems=None,
                    impact_score=1, description=''):
        incident = {
            'id': incident_id,
            'title': title,
            'severity': severity,
            'category': category,
            'detected_time': datetime.datetime.strptime(detected_time, '%Y-%m-%d %H:%M:%S'),
            'resolved_time': datetime.datetime.strptime(resolved_time, '%Y-%m-%d %H:%M:%S') if resolved_time else None,
            'status': status,
            'affected_systems': affected_systems or [],
            'impact_score': impact_score,
            'description': description,
            'duration': None
        }

        if incident['resolved_time']:
            incident['duration'] = (incident['resolved_time'] - incident['detected_time']).total_seconds() / 3600

        self.incidents.append(incident)

    def load_sample_incidents(self):
        sample_incidents = [
            ('INC-001', 'Malware Detection on Workstation', 'High', 'Malware',
             '2024-01-15 09:30:00', '2024-01-15 14:20:00', 'Resolved', ['WS-001', 'WS-002'], 3),
            ('INC-002', 'Suspicious Login Activity', 'Medium', 'Authentication',
             '2024-01-16 14:45:00', '2024-01-16 16:30:00', 'Resolved', ['AD-Server'], 2),
            ('INC-003', 'DDoS Attack Detected', 'Critical', 'Network',
             '2024-01-18 22:15:00', '2024-01-19 03:45:00', 'Resolved', ['Web-Server', 'Load-Balancer'], 5),
            ('INC-004', 'Phishing Email Campaign', 'High', 'Social Engineering',
             '2024-01-20 11:00:00', '2024-01-20 17:30:00', 'Resolved', ['Email-Server'], 4),
            ('INC-005', 'Unauthorized Access Attempt', 'Critical', 'Access Control',
             '2024-01-22 08:20:00', '2024-01-22 12:45:00', 'Resolved', ['Database-Server'], 5),
            ('INC-006', 'Data Exfiltration Alert', 'Critical', 'Data Breach',
             '2024-01-25 16:30:00', '2024-01-26 09:15:00', 'Resolved', ['File-Server', 'Database'], 5),
            ('INC-007', 'Ransomware Indicators', 'Critical', 'Malware',
             '2024-01-28 13:45:00', '2024-01-29 11:30:00', 'Resolved', ['Multiple-Systems'], 5),
            ('INC-008', 'Insider Threat Activity', 'High', 'Insider Threat',
             '2024-02-01 10:15:00', '2024-02-02 14:20:00', 'Resolved', ['HR-Systems'], 4),
            ('INC-009', 'SQL Injection Attempt', 'High', 'Web Application',
             '2024-02-03 19:30:00', '2024-02-03 23:45:00', 'Resolved', ['Web-App-DB'], 3),
            ('INC-010', 'Certificate Expiration', 'Medium', 'Infrastructure',
             '2024-02-05 07:00:00', '2024-02-05 09:30:00', 'Resolved', ['SSL-Certificate'], 2),
            ('INC-011', 'Privilege Escalation', 'High', 'Access Control',
             '2024-02-08 15:20:00', None, 'In Progress', ['Domain-Controller'], 4),
            ('INC-012', 'Anomalous Network Traffic', 'Medium', 'Network',
             '2024-02-10 12:00:00', None, 'Open', ['Firewall-Logs'], 2)
        ]

        for incident_data in sample_incidents:
            self.add_incident(*incident_data)

    def generate_timeline_chart(self):
        if not self.incidents:
            print("No incidents to display")
            return

        fig, ax = plt.subplots(figsize=(16, 10))

        y_positions = {}
        current_y = 0

        for i, incident in enumerate(self.incidents):
            y_positions[incident['id']] = current_y

            start_time = incident['detected_time']
            end_time = incident['resolved_time'] if incident['resolved_time'] else datetime.datetime.now()

            duration = (end_time - start_time).total_seconds() / 3600

            color = self.severity_colors.get(incident['severity'], '#888888')

            rect = Rectangle((mdates.date2num(start_time), current_y - 0.4),
                           mdates.date2num(end_time) - mdates.date2num(start_time), 0.8,
                           facecolor=color, alpha=0.7, edgecolor='black', linewidth=1)
            ax.add_patch(rect)

            ax.text(mdates.date2num(start_time), current_y,
                   f"{incident['id']}: {incident['title'][:30]}...",
                   va='center', ha='left', fontsize=8, weight='bold')

            current_y += 1

        ax.set_ylim(-0.5, len(self.incidents) - 0.5)
        ax.set_xlim(mdates.date2num(min(inc['detected_time'] for inc in self.incidents)) - 0.5,
                   mdates.date2num(max(inc['resolved_time'] if inc['resolved_time'] else datetime.datetime.now()
                                     for inc in self.incidents)) + 0.5)

        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

        ax.set_xlabel('Timeline')
        ax.set_ylabel('Incidents')
        ax.set_title('Security Incident Timeline')
        ax.grid(True, alpha=0.3)

        severity_legend = [plt.Rectangle((0,0),1,1, facecolor=color, alpha=0.7)
                          for color in self.severity_colors.values()]
        ax.legend(severity_legend, self.severity_colors.keys(),
                 loc='upper left', bbox_to_anchor=(1, 1), title='Severity')

        plt.tight_layout()
        plt.show()

    def generate_severity_distribution(self):
        severities = [inc['severity'] for inc in self.incidents]
        severity_counts = pd.Series(severities).value_counts()

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

        colors = [self.severity_colors[sev] for sev in severity_counts.index]
        ax1.pie(severity_counts.values, labels=severity_counts.index,
               autopct='%1.1f%%', colors=colors, startangle=90)
        ax1.set_title('Incident Distribution by Severity')

        ax2.bar(severity_counts.index, severity_counts.values, color=colors)
        ax2.set_title('Incident Count by Severity')
        ax2.set_ylabel('Count')
        ax2.set_xlabel('Severity Level')

        plt.tight_layout()
        plt.show()

    def generate_category_analysis(self):
        categories = [inc['category'] for inc in self.incidents]
        category_counts = pd.Series(categories).value_counts()

        plt.figure(figsize=(12, 8))
        plt.barh(category_counts.index, category_counts.values,
                color=plt.cm.Set3(np.linspace(0, 1, len(category_counts))))
        plt.title('Incidents by Category')
        plt.xlabel('Number of Incidents')
        plt.ylabel('Category')

        for i, v in enumerate(category_counts.values):
            plt.text(v + 0.05, i, str(v), va='center', fontweight='bold')

        plt.tight_layout()
        plt.show()

    def generate_response_time_analysis(self):
        resolved_incidents = [inc for inc in self.incidents if inc['resolved_time']]

        if not resolved_incidents:
            print("No resolved incidents to analyze")
            return

        response_times = [inc['duration'] for inc in resolved_incidents]
        severities = [inc['severity'] for inc in resolved_incidents]

        df = pd.DataFrame({
            'Response_Time': response_times,
            'Severity': severities,
            'Category': [inc['category'] for inc in resolved_incidents]
        })

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

        ax1.hist(response_times, bins=10, alpha=0.7, color='skyblue', edgecolor='black')
        ax1.set_title('Response Time Distribution')
        ax1.set_xlabel('Hours')
        ax1.set_ylabel('Frequency')

        severity_response = df.groupby('Severity')['Response_Time'].mean()
        colors = [self.severity_colors[sev] for sev in severity_response.index]
        ax2.bar(severity_response.index, severity_response.values, color=colors)
        ax2.set_title('Average Response Time by Severity')
        ax2.set_xlabel('Severity')
        ax2.set_ylabel('Average Hours')

        category_response = df.groupby('Category')['Response_Time'].mean().sort_values(ascending=False)
        ax3.barh(category_response.index, category_response.values,
                color=plt.cm.Set2(np.linspace(0, 1, len(category_response))))
        ax3.set_title('Average Response Time by Category')
        ax3.set_xlabel('Average Hours')

        ax4.scatter(df['Response_Time'], df.index,
                   c=[self.severity_colors[sev] for sev in df['Severity']],
                   s=100, alpha=0.7)
        ax4.set_title('Response Time vs Incident Order')
        ax4.set_xlabel('Response Time (Hours)')
        ax4.set_ylabel('Incident Index')

        plt.tight_layout()
        plt.show()

    def generate_status_dashboard(self):
        statuses = [inc['status'] for inc in self.incidents]
        status_counts = pd.Series(statuses).value_counts()

        fig, ax = plt.subplots(figsize=(10, 6))

        colors = [self.status_colors[status] for status in status_counts.index]
        bars = ax.bar(status_counts.index, status_counts.values, color=colors)

        ax.set_title('Incident Status Overview')
        ax.set_xlabel('Status')
        ax.set_ylabel('Count')

        for bar, count in zip(bars, status_counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                   str(count), ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        plt.show()

    def generate_impact_heatmap(self):
        impact_matrix = []
        severity_order = ['Critical', 'High', 'Medium', 'Low', 'Info']
        categories = list(set(inc['category'] for inc in self.incidents))

        for category in categories:
            row = []
            for severity in severity_order:
                count = len([inc for inc in self.incidents
                           if inc['category'] == category and inc['severity'] == severity])
                row.append(count)
            impact_matrix.append(row)

        plt.figure(figsize=(10, 8))
        sns.heatmap(impact_matrix,
                   xticklabels=severity_order,
                   yticklabels=categories,
                   annot=True,
                   fmt='d',
                   cmap='YlOrRd',
                   cbar_kws={'label': 'Incident Count'})

        plt.title('Incident Impact Heatmap: Category vs Severity')
        plt.xlabel('Severity')
        plt.ylabel('Category')
        plt.tight_layout()
        plt.show()

    def print_incident_summary(self):
        print("=== SECURITY INCIDENT TIMELINE SUMMARY ===\n")

        total_incidents = len(self.incidents)
        open_incidents = len([inc for inc in self.incidents if inc['status'] == 'Open'])
        resolved_incidents = len([inc for inc in self.incidents if inc['status'] == 'Resolved'])

        print(f"Total Incidents: {total_incidents}")
        print(f"Open Incidents: {open_incidents}")
        print(f"Resolved Incidents: {resolved_incidents}")

        if resolved_incidents > 0:
            avg_response_time = np.mean([inc['duration'] for inc in self.incidents if inc['duration']])
            print(f"Average Response Time: {avg_response_time:.2f} hours")

        print("\nCritical Incidents:")
        critical_incidents = [inc for inc in self.incidents if inc['severity'] == 'Critical']
        for inc in critical_incidents:
            status_indicator = "🔴" if inc['status'] != 'Resolved' else "✅"
            print(f"  {status_indicator} {inc['id']}: {inc['title']}")

        print("\nRecent Activity (Last 7 Days):")
        recent_cutoff = datetime.datetime.now() - datetime.timedelta(days=7)
        recent_incidents = [inc for inc in self.incidents if inc['detected_time'] >= recent_cutoff]
        for inc in recent_incidents:
            print(f"  • {inc['detected_time'].strftime('%Y-%m-%d %H:%M')} - {inc['title']}")

    def run_comprehensive_analysis(self):
        self.load_sample_incidents()
        self.print_incident_summary()
        self.generate_timeline_chart()
        self.generate_severity_distribution()
        self.generate_category_analysis()
        self.generate_response_time_analysis()
        self.generate_status_dashboard()
        self.generate_impact_heatmap()

def main():
    tracker = SecurityIncidentTracker()
    tracker.run_comprehensive_analysis()

if __name__ == "__main__":
    main()